# 📊 W3: 학생 피드백 & 성적 분석 자동화
**hexa-4 — Week 3** | ⏱️ 60분 | 🎯 성적 분포 시각화 + 학부모 문자 생성

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'],capture_output=True)
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
_n=[f for f in fm.findSystemFonts() if 'Nanum' in f]
if _n: fm.fontManager.addfont(_n[0]); plt.rcParams['font.family']=fm.FontProperties(fname=_n[0]).get_name()
plt.rcParams['axes.unicode_minus']=False
!pip install -q google-generativeai pandas matplotlib
try:
    from google.colab import userdata; API_KEY=userdata.get('GEMINI_API_KEY')
except: API_KEY=input('GEMINI_API_KEY: ')
import google.generativeai as genai; genai.configure(api_key=API_KEY)
model=genai.GenerativeModel('gemini-2.0-flash'); print('✅ 준비 완료')


## Step 1: 학생 성적 입력 (✏️)

In [ ]:
import pandas as pd
STUDENTS=pd.DataFrame([
    {'이름':'김민준','점수':92,'과목':'수학'},{'이름':'이서연','점수':78,'과목':'수학'},
    {'이름':'박도현','점수':65,'과목':'수학'},{'이름':'최지아','점수':88,'과목':'수학'},
    {'이름':'정예준','점수':45,'과목':'수학'},{'이름':'강하은','점수':95,'과목':'수학'},
    {'이름':'윤서준','점수':72,'과목':'수학'},{'이름':'임나은','점수':83,'과목':'수학'},
])
print(f'학생 수: {len(STUDENTS)}명 | 평균: {STUDENTS["점수"].mean():.1f}점')


## Step 2: 성적 분포 시각화

In [ ]:
import matplotlib.pyplot as plt
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,4))
ax1.hist(STUDENTS['점수'],bins=5,color='steelblue',edgecolor='white')
ax1.axvline(STUDENTS['점수'].mean(),color='red',linestyle='--',label=f'평균:{STUDENTS["점수"].mean():.1f}')
ax1.set_title('성적 분포'); ax1.legend()
colors=['#e74c3c' if s<60 else '#f39c12' if s<80 else '#2ecc71' for s in STUDENTS['점수']]
ax2.bar(STUDENTS['이름'],STUDENTS['점수'],color=colors)
ax2.set_title('학생별 점수'); ax2.tick_params(axis='x',rotation=30)
plt.tight_layout(); plt.savefig('grade_chart.png',dpi=150,bbox_inches='tight'); plt.show()
print(f'✅최고:{STUDENTS["점수"].max()} 최저:{STUDENTS["점수"].min()} 평균:{STUDENTS["점수"].mean():.1f}')


## Step 3: 학부모 문자 자동 생성

In [ ]:
def gen_msg(name, score):
    level='우수' if score>=90 else '양호' if score>=70 else '노력필요'
    p=f"{name} 학생 점수: {score}점 ({level}). 학부모 문자 100자 이내로 생성. 따뜻하고 구체적으로."
    return model.generate_content(p).text.strip()[:100]
STUDENTS['학부모문자']=[gen_msg(r['이름'],r['점수']) for _,r in STUDENTS.iterrows()]
print('✅ 생성 완료')
for _,r in STUDENTS.iterrows(): print(f"[{r['이름']}({r['점수']})] {r['학부모문자']}\n")


## Step 4: CSV + 차트 다운로드

In [ ]:
STUDENTS.to_csv('grade_feedback.csv',index=False,encoding='utf-8-sig')
from google.colab import files
files.download('grade_feedback.csv'); files.download('grade_chart.png')
print('✅ 완료!')
